In [318]:
# Import public packages and functions
import os
import pandas as pd
import numpy as np
import sys
import json
from pathlib import Path
from scipy import signal

import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp

import warnings
warnings.filterwarnings("ignore")

# inserting the lib folder to the compiler
sys.path.insert(0, './lib')
sys.path.insert(0, './utils/')

import utils_io, utils_trgc, utils_plotting

from lib_data import DATA_IO

In [320]:
PATH_CURR = os.path.abspath(os.curdir)    # current code
PATH      = (str(Path(PATH_CURR).parent)) # data repository: upper directory where datasets situated
fs        = 2048

# 1. Data Load & Preprocess

In [324]:
taps_gc       = pd.read_pickle(DATA_IO.path_events + "granger_causality/ipsilateral_connectivity.pkl")

In [341]:
from scipy.interpolate import griddata
from pyvistaqt import BackgroundPlotter
import pyvista as pv
from scipy.interpolate import Rbf
from scipy.spatial import cKDTree

def plot_gc_connectivity_for_ECOG_channels(df, state, segment, frequency_band, vmin, vmax, cortical_area=None, only_significant=False):

    #######################################################################################
    # STEP 1: filter the dataset
    #######################################################################################
    
    df_filtered    = df[(df.state==state) & (df.segment==segment) & (df.direction=="ecog->lfp")].copy()
    # filter by significance / if indicated
    if(only_significant == True): df_filtered = df_filtered[df_filtered[f"pvalue_{frequency_band}"] <= 0.05].copy()

    #######################################################################################
    # STEP 2: get the mean connectivity for each ECOG channel across LFP channels 
    #######################################################################################
    
    # group by patient, state, source channel
    group_cols          = ["patient","state","segment","source_type","source_hemisphere","source_channel"]

    # taking the mean across LFP channels for the ECOG channels
    df_filtered_mean    = df_filtered.groupby(group_cols)[f"gc_{frequency_band}"].mean().reset_index()

    # read ECOG channel coordinates
    MNI_ECoG_channels   = pd.read_pickle(DATA_IO.path_coordinates + "MNI_ECoG_channels.pkl")
    MNI_ECoG_channels.x = MNI_ECoG_channels.x.abs()
    mni_channels        = MNI_ECoG_channels.rename(columns={"hemisphere": "source_hemisphere","channel": "source_channel"})
    mni_channels        = mni_channels[["patient","source_hemisphere","source_channel","x","y","z","AAL3_cortex"]]
    
    # merge to data frame by mapping hemisphere + ECOG hemisphere + ECOG channel
    df_plot_data        = df_filtered_mean.merge(mni_channels, on=["patient", "source_hemisphere", "source_channel"], how="left")

    if(cortical_area is not None):
        df_plot_data        = df_plot_data[df_plot_data.AAL3_cortex==cortical_area]

    #######################################################################################
    # STEP 3: plot the connectivity on the surface of cortex 
    #######################################################################################

    coords                            = df_plot_data[['x', 'y', 'z']].to_numpy()
    values                            = df_plot_data[f"gc_{frequency_band}"].to_numpy()
    
    # load meshes
    cortex_mesh                       = utils_io.load_cortical_atlas_meshes()
    cortex_right                      = cortex_mesh["right_hemisphere"] # we mapped all left hemisphere recordings to right
    
    # cortical mesh vertices
    radius                            = 20
    mesh_vertices                     = cortex_right.points
    
    # RBF interpolation
    rbf                               = Rbf(coords[:,0], coords[:,1], coords[:,2], values, function='multiquadric', smooth=2)
    values_interp                     = rbf(mesh_vertices[:,0], mesh_vertices[:,1], mesh_vertices[:,2])
    
    # mask vertices outside radius
    tree                              = cKDTree(coords)
    distances, _                      = tree.query(mesh_vertices)
    values_interp[distances > radius] = np.nan
    cortex_right[frequency_band]      = values_interp
    cortex_smoothed                   = cortex_right.smooth(n_iter=100, relaxation_factor=0.01)
    
    # Plot
    plotter = BackgroundPlotter()
    plotter.add_mesh(cortex_smoothed, color='white', scalars=frequency_band, cmap='Reds', opacity=1, 
                     clim=(vmin, vmax), scalar_bar_args={"color": "black", "n_labels": 5,"fmt": "%.4f"}, smooth_shading=True)
    
    # Add electrodes
    for coor in coords:
        plotter.add_mesh(pv.Sphere(radius=2, center=[coor[0], coor[1], coor[2]+5]), color='lightgrey', smooth_shading=True)
    
    plotter.background_color = 'white'
    plotter.add_text(f'{state} - {frequency_band} - {segment}', position='upper_right', font_size=16, color='black')
    plotter.view_vector((0,0,1))
    plotter.add_light(pv.Light(position=(0,0,1), color='white', intensity=0.6))
    plotter.screenshot(f'{state}-{frequency_band}-{segment}.png', transparent_background=False)
    plotter.show()

In [391]:
vmin=0.005; vmax=0.015
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "pre_event" , frequency_band = "gamma", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "event"     , frequency_band = "gamma", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "post_event", frequency_band = "gamma", vmin=vmin, vmax=vmax)

In [397]:
vmin=0.005; vmax=0.015
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "pre_event" , frequency_band = "beta_high", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "event"     , frequency_band = "beta_high", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "post_event", frequency_band = "beta_high", vmin=vmin, vmax=vmax)

In [399]:
vmin=0.005; vmax=0.015
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "pre_event" , frequency_band = "beta_low", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "event"     , frequency_band = "beta_low", vmin=vmin, vmax=vmax)
plot_gc_connectivity_for_ECOG_channels(taps_gc, state = "LID", segment = "post_event", frequency_band = "beta_low", vmin=vmin, vmax=vmax)

In [243]:
plot_dt = plot_gc_connectivity_for_ECOG_channels(df_net, state = "LID", segment = "event", frequency_band = "gamma")

In [279]:


def compute_net_gc_directionality(df):
    
    freq_bands         = ["theta","alpha","beta_low","beta_high","gamma","gamma_III"]
    unique_combination = df[["patient","state","segment"]].drop_duplicates()
    
    # unique combinations
    results = []
    
    for _, combination in unique_combination.iterrows():
        
        patient = combination.patient
        state   = combination.state
        segment = combination.segment
        df_sub  = df[(df.patient==patient) & (df.state==state) & (df.segment==segment)] # subset for this patient/state/segment

        # Unique LFP-ECoG channel pairs
        lfp_channels  = df_sub[df_sub.source_type=="LFP"]["source_channel"].unique()
        ecog_channels = df_sub[df_sub.source_type=="ECOG"]["source_channel"].unique()

        for lfp_ch in lfp_channels:
            for ecog_ch in ecog_channels:
                
                # Select the two rows corresponding to this pair
                df_pair = df_sub[((df_sub.source_type=="LFP")  & (df_sub.source_channel==lfp_ch) & (df_sub.target_type=="ECOG") & (df_sub.target_channel==ecog_ch)) |
                                 ((df_sub.source_type=="ECOG") & (df_sub.source_channel==ecog_ch) & (df_sub.target_type=="LFP") & (df_sub.target_channel==lfp_ch))]
                
                if len(df_pair) != 2: continue  # skip incomplete pairs as a sanity check

                # Choose the ECOG->LFP row as the output row (keep structure)
                row_ecog_to_lfp = df_pair.loc[df_pair.source_type=="ECOG"].copy()

                # Compute net GC for each frequency band
                for band in freq_bands:
                    gc_ecog_to_lfp                = df_pair.loc[df_pair.source_type=="ECOG", f"gc_{band}"].values[0]
                    gc_lfp_to_ecog                = df_pair.loc[df_pair.source_type=="LFP", f"gc_{band}"].values[0]
                    row_ecog_to_lfp[f"gc_{band}"] = gc_ecog_to_lfp - gc_lfp_to_ecog

                results.append(row_ecog_to_lfp)
                
    return pd.concat(results, ignore_index=True)


In [189]:
df_net = compute_net_gc_directionality(taps_gc)

In [209]:
df_net

,source_type,source_hemisphere,source_channel,source_index,target_type,target_hemisphere,target_channel,target_index,patient,state,...,gc_alpha,gc_beta_low,gc_beta_high,gc_gamma,gc_gamma_III,pvalue_theta,pvalue_alpha,pvalue_beta_low,pvalue_beta_high,pvalue_gamma
0,ECOG,left,02-01,5,LFP,left,02-01,0,009,MED-OFF,...,-0.010742,0.003113,0.007446,0.004527,0.002707,1.0,0.0,0.0,0.0,0.0
1,ECOG,left,03-02,6,LFP,left,02-01,0,009,MED-OFF,...,-0.001169,-0.001962,0.002894,-0.003227,-0.003043,1.0,1.0,1.0,0.0,1.0
2,ECOG,left,04-03,7,LFP,left,02-01,0,009,MED-OFF,...,-0.005705,0.002743,-0.007558,-0.000640,-0.000498,0.0,1.0,0.0,0.0,1.0
3,ECOG,left,05-04,8,LFP,left,02-01,0,009,MED-OFF,...,-0.000300,0.007409,-0.000391,0.002140,0.004886,1.0,1.0,0.0,1.0,0.0
4,ECOG,left,06-05,9,LFP,left,02-01,0,009,MED-OFF,...,-0.006795,0.000474,0.003019,0.003832,0.008083,1.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1548,ECOG,right,02-01,2,LFP,right,07-04,1,023,LID,...,0.034739,0.016924,0.019463,-0.003282,-0.005394,0.0,0.0,0.0,0.0,0.0
1549,ECOG,right,03-02,3,LFP,right,07-04,1,023,LID,...,-0.007741,-0.002903,0.009449,-0.001891,-0.008575,0.0,0.0,1.0,0.0,1.0
1550,ECOG,right,04-03,4,LFP,right,07-04,1,023,LID,...,0.003162,0.008693,-0.007671,0.002145,-0.000926,0.0,0.0,0.0,0.0,0.0
1551,ECOG,right,05-04,5,LFP,right,07-04,1,023,LID,...,-0.001760,-0.002300,-0.019517,0.002832,-0.003634,0.0,1.0,0.0,0.0,1.0


In [199]:
plot_gc_connectivity_for_ECOG_channels(df_net, state = "MED-OFF", segment = "pre_event", frequency_band = "gamma")
plot_gc_connectivity_for_ECOG_channels(df_net, state = "MED-ON" , segment = "pre_event", frequency_band = "gamma")
plot_gc_connectivity_for_ECOG_channels(df_net, state = "LID"    , segment = "pre_event", frequency_band = "gamma")